# RAG

Implement a base RAG module in DSPy. 
Given a question, retrieve the top-k documents in a list of HTML documents, then pass them as context to an LLM.

Refer to https://dspy.ai/tutorials/rag/. 


In [1]:
import dspy
from sentence_transformers import SentenceTransformer

# Load an extremely efficient local model for retrieval
model = SentenceTransformer("sentence-transformers/static-retrieval-mrl-en-v1", device="cpu")

# Create an embedder using the model's encode method
embedder = dspy.Embedder(model.encode)

# Traverse a directory and read html files - extract text from the html files
import os
from bs4 import BeautifulSoup
def read_html_files(directory):
    texts = []
    for filename in os.listdir(directory):
        if filename.endswith(".html"):
            with open(os.path.join(directory, filename), 'r', encoding='utf-8') as file:
                soup = BeautifulSoup(file, 'html.parser')
                texts.append(soup.get_text())
    return texts

In [2]:
corpus = read_html_files("../PragmatiCQA-sources/The Legend of Zelda")
print(f"Loaded {len(corpus)} documents. Will encode them below.")

Loaded 406 documents. Will encode them below.


In [12]:
# Parameters for the retriever
max_characters = 10000  # for truncating >99th percentile of documents
topk_docs_to_retrieve = 3  # number of documents to retrieve per search query

search = dspy.retrievers.Embeddings(embedder=embedder, corpus=corpus, k=topk_docs_to_retrieve)



In [4]:
# lm = dspy.LM('ollama_chat/devstral', api_base='http://localhost:11434', api_key='')
lm = dspy.LM('xai/grok-3-mini')
dspy.configure(lm=lm)

In [7]:
with open("grok_key.ini") as f:
    for line in f:
        if "XAI_API_KEY" in line and not line.strip().startswith("#"):
            key_value = line.strip().split("=")
            if len(key_value) == 2:
                os.environ["XAI_API_KEY"] = key_value[1].split()[0]

In [8]:
class RAG(dspy.Module):
    def __init__(self):
        self.respond = dspy.ChainOfThought('context, question -> response')

    def forward(self, question):
        context = search(question).passages
        return self.respond(context=context, question=question)
    
rag = RAG()

In [9]:
answer = rag(question="What is the main plot of The Legend of Zelda?")  # Example query

print(answer.response)  # Print the response from the RAG model

The main plot of *The Legend of Zelda* revolves around a young hero named Link who embarks on a quest to save the kingdom of Hyrule. An evil army led by Ganon, the Prince of Darkness, steals the Triforce of Power and seeks the Triforce of Wisdom. Princess Zelda splits the Triforce of Wisdom into eight fragments and hides them to prevent Ganon from obtaining it. She sends her nursemaid, Impa, to find a brave warrior. Link rescues Impa from Ganon's henchmen, learns of the crisis, and sets out to collect the eight fragments, reassemble the Triforce of Wisdom, and confront Ganon in his lair to rescue Princess Zelda and restore peace to Hyrule.


In [13]:
q = 'What year did the Legend of Zelda come out?' 

print(rag(question=q).response)

The Legend of Zelda was first released in 1986.


In [14]:
dspy.inspect_history()





[2025-08-19T11:01:00.408677]

System message:

Your input fields are:
1. `context` (str): 
2. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `response` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## context ## ]]
{context}

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## response ## ]]
{response}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `context`, `question`, produce the fields `response`.


User message:

[[ ## context ## ]]
[1] «««
    
    
    
    
    
    
    
    
          This article is a short summary of Shigeru Miyamoto.
          NintendoWiki features
          
           a more in-depth article
          
          .
         
    
    
    
    
    
    
    
          Shigeru Miyamoto
          宮本 茂
          みやもと しげる
         
    
    
    
          Current Position
         
    
    
       